# Job Recommender Model Training
This notebook trains the TF-IDF and BERT models on the full dataset using a cloud GPU.

In [ ]:
# Install dependencies if running on Colab
%pip install sentence-transformers pandas numpy scikit-learn joblib torch gdown

### Step 0: Download Dataset via gdown
Since we are running in VS Code with a Colab remote, Google Drive mounting is unavailable. 
We will instead download the dataset directly from your public Google Drive link using `gdown`.

In [ ]:
import os

if not os.path.exists("data"):
    os.makedirs("data")

if not os.path.exists("data/job_descriptions.csv"):
    print("Downloading dataset from Google Drive...")
    !gdown --id 1mMU64Zx25Y3sanvCpCQS1sSjmhQnStOm -O data/job_descriptions.csv
else:
    print("Dataset already exists at data/job_descriptions.csv")

print("\n✅ Dataset ready!")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import re
import torch
from scipy.sparse import vstack
import gc

from sentence_transformers import SentenceTransformer

# -----------------------------------------------------
# HELPER FUNCTIONS (originally from src module)
# -----------------------------------------------------
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9+#.\s]', ' ', text)
    text = text.replace("c plus plus", "c++")
    text = text.replace("c hash", "c#")
    text = text.replace("node js", "node.js")
    text = text.replace("asp.net", "asp net")
    text = re.sub(r'\baiml\b', 'artificial intelligence machine learning', text)
    text = re.sub(r'\bnlp\b', 'natural language processing', text)
    return text.strip()

_bert_model = None
def get_bert_model(model_name='all-MiniLM-L6-v2'):
    global _bert_model
    if _bert_model is None:
        try:
            import torch
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        except ImportError:
            device = 'cpu'
        _bert_model = SentenceTransformer(model_name, device=device)
    return _bert_model

def get_bert_embeddings(corpus):
    model = get_bert_model()
    if isinstance(corpus, str):
        corpus = [corpus]
    batch_size = 32
    embeddings = []
    for i in range(0, len(corpus), batch_size):
        batch = corpus[i:i+batch_size]
        batch_emb = model.encode(batch, show_progress_bar=False)
        embeddings.append(batch_emb)
    embeddings = np.vstack(embeddings)
    return embeddings
# -----------------------------------------------------

# =====================================================
# CONFIGURATION - Update DATA_PATH to where your CSV is!
# =====================================================

DATA_PATH = "data/job_descriptions.csv"

ARTIFACTS_DIR = "artifacts"
MAX_TOTAL_JOBS = 500000 
SAMPLE_SIZE_FOR_FIT = 5000
CHUNK_SIZE = 512
MAX_FEATURES = 2000 

if not os.path.exists(ARTIFACTS_DIR):
    os.makedirs(ARTIFACTS_DIR)

def get_combined_text(df):
    """Combine job features with description truncation."""
    df = df.copy()
    df['Job Title'] = df['Job Title'].fillna('')
    skill_col = 'skills' if 'skills' in df.columns else 'Skills'
    df[skill_col] = df[skill_col].fillna('')
    df['Job Description'] = df['Job Description'].fillna('')
    df['Job Description'] = df['Job Description'].str[:1000]
    return (
        df['Job Title'] + " " +
        df[skill_col] + " " +
        df['Job Description']
    ).apply(preprocess_text)


In [ ]:
print("--- Starting Offline Training Pipeline ---")
if torch.cuda.is_available():
    print(f"[INFO] GPU detected: {torch.cuda.get_device_name(0)}. BERT will use CUDA.")
else:
    print("[INFO] No GPU detected. BERT will use CPU. Training may be slow.")

# Step 1.5: Load and sample data
print(f"Phase 1.5: Loading dataset from {DATA_PATH}...")
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"\n❌ Could not find {DATA_PATH}. Please run the cell above to download it via gdown.")

all_data = pd.read_csv(DATA_PATH)
print(f"  Total rows available: {len(all_data)}")

if len(all_data) > MAX_TOTAL_JOBS:
    print(f"  Sampling {MAX_TOTAL_JOBS} jobs...")
    all_data = all_data.sample(n=MAX_TOTAL_JOBS, random_state=42)
print(f"  Using dataset: {len(all_data)} jobs")


In [ ]:
# Step 2: Fit TF-IDF on a sample to build vocabulary
print(f"Phase 2: Fitting TF-IDF...")
sample_df = all_data.head(SAMPLE_SIZE_FOR_FIT) if len(all_data) > SAMPLE_SIZE_FOR_FIT else all_data
combined_sample = get_combined_text(sample_df)

tfidf = TfidfVectorizer(
    max_features=MAX_FEATURES,
    ngram_range=(1, 2),
    stop_words='english'
)
tfidf.fit(combined_sample)
joblib.dump(tfidf, os.path.join(ARTIFACTS_DIR, "tfidf.pkl"))
print("Saved tfidf.pkl")


In [ ]:
# Step 3: Transform dataset in chunks
print("Phase 3: Transforming dataset in chunks...")
all_vectors = []

processed_count = 0
for chunk_num, i in enumerate(range(0, len(all_data), CHUNK_SIZE)):
    chunk = all_data.iloc[i:i+CHUNK_SIZE].copy()
    
    if chunk_num % 10 == 0:
        print(f"  TF-IDF chunk {chunk_num}: {i} to {min(i+CHUNK_SIZE, len(all_data))}...")
    
    combined_chunk = get_combined_text(chunk)
    vectors = tfidf.transform(combined_chunk)
    all_vectors.append(vectors)
    
    processed_count += len(chunk)

# Step 4: Stack TF-IDF vectors
print("Phase 4: Stacking TF-IDF vectors...")
final_tfidf_vectors = vstack(all_vectors)
print(f"  TF-IDF matrix shape: {final_tfidf_vectors.shape}")


In [ ]:
# Step 5: Keep metadata with only essential columns
print("Phase 5: Preparing metadata...")

if 'Location' not in all_data.columns:
    city_col    = 'location' if 'location' in all_data.columns else None
    country_col = 'Country'  if 'Country'  in all_data.columns else None
    if city_col and country_col:
        all_data['Location'] = (
            all_data[city_col].fillna('').astype(str).str.strip() + ', ' +
            all_data[country_col].fillna('').astype(str).str.strip()
        ).str.strip(', ')
    elif city_col:
        all_data['Location'] = all_data[city_col].fillna('').astype(str).str.strip()
    elif country_col:
        all_data['Location'] = all_data[country_col].fillna('').astype(str).str.strip()
    else:
        all_data['Location'] = 'Remote'

cols_to_keep = ['Job Title', 'Company', 'Location', 'skills', 'Job Description', 'Experience']
available_cols = [c for c in cols_to_keep if c in all_data.columns]

if 'id' not in all_data.columns:
    all_data['id'] = range(len(all_data))
    all_data['id'] = all_data['id'].astype(str)

if 'id' not in available_cols:
    available_cols.insert(0, 'id')

metadata_df = all_data[available_cols].reset_index(drop=True)
print(f"  Metadata prepared: {len(metadata_df)} records")


In [ ]:
# Step 6: Generate Semantic Embeddings (Sentence-BERT)
print("Phase 6: Generating BERT Embeddings (chunked)...")

bert_vectors_list = []

for batch_num, i in enumerate(range(0, len(metadata_df), CHUNK_SIZE)):
    batch_start = i
    batch_end = min(i + CHUNK_SIZE, len(metadata_df))
    if batch_num % 10 == 0:
        print(f"  BERT batch {batch_num}: {batch_start} to {batch_end}/{len(metadata_df)}...")
    
    batch_df = metadata_df.iloc[batch_start:batch_end]
    batch_texts = get_combined_text(batch_df).tolist()
    
    # Generate embeddings for this batch only
    batch_embeddings = get_bert_embeddings(batch_texts)
    
    # SAVE TO DISK immediately to free RAM
    chunk_file = os.path.join(ARTIFACTS_DIR, f"bert_chunk_{batch_num:04d}.npy")
    np.save(chunk_file, batch_embeddings)
    
    bert_vectors_list.append(chunk_file)


In [ ]:
# Step 7: Reload and stack BERT vectors from disk
print("Phase 7: Stacking BERT embeddings from disk...")
bert_vectors = np.vstack([np.load(f) for f in bert_vectors_list])
print(f"  BERT vectors shape: {bert_vectors.shape}")

# Explicit memory cleanup
if 'all_vectors' in locals(): del all_vectors
if 'combined_sample' in locals(): del combined_sample
gc.collect()


In [ ]:
# Step 8: Saving final artifacts
print("Phase 8: Saving final artifacts...")
joblib.dump(tfidf, os.path.join(ARTIFACTS_DIR, "tfidf.pkl"))
joblib.dump(final_tfidf_vectors, os.path.join(ARTIFACTS_DIR, "job_vectors.pkl"))
joblib.dump(bert_vectors, os.path.join(ARTIFACTS_DIR, "bert_job_vectors.pkl"))
metadata_df.to_pickle(os.path.join(ARTIFACTS_DIR, "jobs.pkl"))

print(f"\n\u2705 Saved artifacts with {len(metadata_df)} records.")
print(f"   - TF-IDF vectors: {final_tfidf_vectors.shape}")
print(f"   - BERT vectors: {bert_vectors.shape}")
print("--- Training Pipeline Complete! ---")

print("\n======================================================")
print("✅ IMPORTANT: Because you are connected via VS Code,")
print("you MUST manually download your generated artifacts.")
print("Open the File Explorer on the left, right-click the")
print("'artifacts' folder, and download them back to")
print("your local machine before closing this session!")
print("======================================================")


### Step 9: Post-Training Automated Evaluation (Zero-Shot Realistic Queries)
This checks how well the system handles realistic short resumes where vocabulary doesn't perfectly match the massive job descriptions.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def dcg_at_k(relevances, k):
    relevances = np.asarray(relevances, dtype=float)[:k]
    if relevances.size:
        return np.sum(relevances / np.log2(np.arange(2, relevances.size + 2)))
    return 0.

def precision_at_k(relevances, k):
    return np.sum(relevances[:k]) / k if k > 0 else 0.

def recall_at_k(relevances, total_relevant, k):
    return np.sum(relevances[:k]) / total_relevant if total_relevant > 0 else 0.

# Simple Recommenders to bypass loading complex src rules
def recommend_jobs_isolated(resume_text, df, tfidf, job_vectors, bert_job_vectors=None, k=50):
    res_tfidf = tfidf.transform([resume_text])
    tfidf_score = cosine_similarity(res_tfidf, job_vectors).flatten()
    
    if bert_job_vectors is not None:
        model = get_bert_model()
        res_bert = model.encode([resume_text], show_progress_bar=False)
        sem_score = cosine_similarity(res_bert, bert_job_vectors).flatten()
        df['score'] = (0.7 * sem_score) + (0.3 * tfidf_score)
    else:
        df['score'] = tfidf_score
        
    top_candidates = df.nlargest(k, 'score')
    return top_candidates['Job Title'].tolist()

TEST_CASES = [
    # Technical
    {"resume": "python machine learning data science", "expected": ["data scientist", "ml engineer", "machine learning"]},
    {"resume": "react javascript frontend html css typescript", "expected": ["frontend", "ui developer", "react developer"]},
    {"resume": "java spring boot backend hibernate", "expected": ["backend", "java developer"]},
    {"resume": "docker kubernetes devops aws ci cd", "expected": ["devops", "infrastructure", "sre", "cloud"]},
    {"resume": "pytorch fastapi ai algorithms", "expected": ["ml engineer", "data scientist", "ai engineer"]},
    {"resume": "sql analytics powerbi tableau", "expected": ["data analyst", "business intelligence", "bi analyst"]},
    # Non-Technical
    {"resume": "b2b sales cold calling crm negotiation", "expected": ["sales", "account executive", "business development"]},
    {"resume": "digital marketing seo sem content strategy", "expected": ["marketing", "seo", "content", "social media"]},
    {"resume": "human resources recruiting payroll employee relations", "expected": ["hr manager", "human resources", "recruiter", "talent"]},
    {"resume": "financial modeling excel accounting gaap CPA", "expected": ["financial analyst", "accountant", "finance", "controller"]},
    {"resume": "customer satisfaction resolving complaints zendesk", "expected": ["customer service", "customer support", "customer experience"]},
    {"resume": "project lifecycle agile scrum stakeholder management", "expected": ["project manager", "scrum master", "product manager"]}
]

print("\n🧬 Running Zero-Shot Evaluation Matrix on 100k Database...")
results = []
k_10 = 10
k_50 = 50
for case in TEST_CASES:
    res_text = case["resume"]
    expected = [role.lower() for role in case["expected"]]
    
    # Precisely calculate how many jobs in the 100k DB match any of our "expected" strings to calculate Total Recall pool
    total_rel = sum(1 for t in metadata_df['Job Title'] if any(exp in str(t).lower() for exp in expected))
    if total_rel == 0: 
        total_rel = 1
    
    old_rec = recommend_jobs_isolated(res_text, metadata_df.copy(), tfidf, final_tfidf_vectors, bert_job_vectors=None, k=k_50)
    new_rec = recommend_jobs_isolated(res_text, metadata_df.copy(), tfidf, final_tfidf_vectors, bert_job_vectors=bert_vectors, k=k_50)
    
    old_rel_50 = [1 if any(exp in str(t).lower() for exp in expected) else 0 for t in old_rec]
    new_rel_50 = [1 if any(exp in str(t).lower() for exp in expected) else 0 for t in new_rec]
    
    idl_dcg_10 = dcg_at_k([1] * min(k_10, total_rel), k_10)
    
    results.append({
        "resume": res_text,
        "old_precision": precision_at_k(old_rel_50, k_10), 
        "new_precision": precision_at_k(new_rel_50, k_10),
        "old_recall": recall_at_k(old_rel_50, total_rel, k_10), 
        "new_recall": recall_at_k(new_rel_50, total_rel, k_10),
        "old_recall_50": recall_at_k(old_rel_50, total_rel, k_50), 
        "new_recall_50": recall_at_k(new_rel_50, total_rel, k_50),
        "old_ndcg": dcg_at_k(old_rel_50, k_10) / idl_dcg_10 if idl_dcg_10 > 0 else 0,
        "new_ndcg": dcg_at_k(new_rel_50, k_10) / idl_dcg_10 if idl_dcg_10 > 0 else 0
    })

rdf = pd.DataFrame(results)
print("\n" + "="*90)
print("📊 REALISTIC EVALUATION RESULTS (K=10 and K=50 Top Matches)")
print("="*90)
print(f"{'Metric':<20} | {'Old TF-IDF Engine':<18} | {'NEW HYBRID ENGINE':<18} | Improvement")
print("-" * 90)
for metric, old_val, new_val in [("Precision@10", rdf['old_precision'].mean(), rdf['new_precision'].mean()),
                                 ("Recall@10", rdf['old_recall'].mean(), rdf['new_recall'].mean()),
                                 ("Recall@50", rdf['old_recall_50'].mean(), rdf['new_recall_50'].mean()),
                                 ("NDCG@10", rdf['old_ndcg'].mean(), rdf['new_ndcg'].mean())]:
    print(f"{metric:<20} | {old_val*100:6.2f}%            | {new_val*100:6.2f}%            | {((new_val - old_val)/(old_val + 1e-8))*100:+.1f}%")
print("="*90)
print("\nCASE STUDIES (Why Hybrid Wins):")
for idx, row in rdf.iterrows():
    if row['new_precision'] > row['old_precision'] or row['new_recall_50'] > row['old_recall_50']:
        print(f"\nQuery: '{row['resume']}'")
        if row['new_precision'] > row['old_precision']:
            print(f"  [Top 10 Accuracy] TF-IDF got {row['old_precision']*100:.0f}% precise / Hybrid got {row['new_precision']*100:.0f}% precise 🔥")
        if row['new_recall_50'] > row['old_recall_50']:
            print(f"  [Deep Retrieval @ 50] TF-IDF found {row['old_recall_50']*100:.1f}% match coverage / Hybrid found {row['new_recall_50']*100:.1f}% match coverage 🔥")
